In [14]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Reshape, Conv1D, MaxPooling1D, Flatten, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE

# ==================================================================================
# 1. Utilize Libraries/Dataset
# This function imports all necessary libraries and loads the diabetes dataset
# ==================================================================================
def import_libraries_and_load_data(file_path):
    """
    Import necessary libraries and load the diabetes dataset

    Parameters:
    file_path (str): Path to the CSV file

    Returns:
    pd.DataFrame: Loaded dataset

    Results:
    - Displays first 5 rows of the dataset
    - Shows dataset information (columns, datatypes, etc.)
    - Displays basic statistics (mean, std, min, max, etc.)
    """
    # Set plotting style
    plt.style.use('ggplot')
    sns.set(style="whitegrid")

    # Load dataset
    df = pd.read_csv(file_path)

    # Display first few rows
    print("First 5 rows of the dataset:")
    print(df.head())

    # Display dataset info
    print("\nDataset Information:")
    print(df.info())

    # Display basic statistics
    print("\nBasic Statistics:")
    print(df.describe())

    return df

In [15]:
# ==================================================================================
# 2.1 Generate EDA Visualizations
# This function creates distribution plots, correlation heatmap, and feature importance
# ==================================================================================
def create_eda_visualizations(df):
    """
    Generate exploratory data analysis visualizations

    Parameters:
    df (pd.DataFrame): The diabetes dataset

    Results:
    - Creates distribution plots comparing features between diabetic and non-diabetic patients
    - Generates correlation heatmap to identify relationships between features
    - Visualizes feature importance using correlation with target
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Create a figure for the distribution plots
    plt.figure(figsize=(20, 15))

    # 1. Distribution plots comparing features between diabetic and non-diabetic patients
    # Selecting numerical columns for distribution analysis
    numerical_cols = ['Age', 'Glucose', 'BMI', 'BloodPressure', 'HbA1c']

    for i, col in enumerate(numerical_cols):
        plt.subplot(3, 2, i+1)
        sns.histplot(data=df, x=col, hue='Diabetes', kde=True, element="step")
        plt.title(f'Distribution of {col} by Diabetes Status')

    plt.tight_layout()
    plt.savefig('distribution_plots.png')
    plt.close()
    print("Distribution plots saved as 'distribution_plots.png'")

    # 2. Correlation heatmap
    plt.figure(figsize=(12, 10))
    numeric_df = df.select_dtypes(include=['float64', 'int64'])
    correlation = numeric_df.corr()
    mask = np.triu(correlation)
    sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", mask=mask)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png')
    plt.close()
    print("Correlation heatmap saved as 'correlation_heatmap.png'")

    # 3. Feature importance visualization
    plt.figure(figsize=(10, 6))
    # Calculate correlation with target
    feature_importance = abs(numeric_df.drop('Diabetes', axis=1).corrwith(numeric_df['Diabetes']))
    feature_importance = feature_importance.sort_values(ascending=False)
    sns.barplot(x=feature_importance.values, y=feature_importance.index)
    plt.title('Feature Importance (Correlation with Diabetes)')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    print("Feature importance visualization saved as 'feature_importance.png'")

    # Return feature importance for later use
    return feature_importance

In [16]:
# ==================================================================================
# 2.2 Analyze Data Quality
# This function checks for missing values, zeros, and outliers
# ==================================================================================
def analyze_data_quality(df):
    """
    Analyze data quality by checking for missing/zero values and identifying outliers

    Parameters:
    df (pd.DataFrame): The diabetes dataset

    Returns:
    pd.DataFrame: DataFrame with replaced zero values

    Results:
    - Reports missing values in each column
    - Identifies zero values in columns where zeros don't make sense
    - Replaces zeros with median values
    - Identifies outliers using IQR method
    """
    # Check for missing values
    print("Missing values in each column:")
    missing_values = df.isnull().sum()
    print(missing_values)

    # Handle missing values if any
    if missing_values.sum() > 0:
        print("\nReplacing missing values with column median...")
        df_processed = df.copy()
        for col in df.columns:
            if missing_values[col] > 0:
                median_value = df[col].median()
                df_processed[col].fillna(median_value, inplace=True)
                print(f"Replaced {missing_values[col]} NaN values in {col} with median: {median_value:.2f}")
    else:
        df_processed = df.copy()
        print("No missing values found in the dataset.")

    # Check for zero values in columns where zero doesn't make sense
    zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    for col in zero_columns:
        if col in df.columns:
            zero_count = (df_processed[col] == 0).sum()
            print(f"Number of zeros in {col}: {zero_count} ({zero_count/len(df_processed)*100:.2f}%)")

    # Handle zero values by replacing with median of non-zero values
    for col in zero_columns:
        if col in df.columns:
            # Replace zeros with median of non-zero values
            median_value = df_processed[df_processed[col] != 0][col].median()
            df_processed.loc[df_processed[col] == 0, col] = median_value
            print(f"Replaced zeros in {col} with median: {median_value:.2f}")

    # Ensure there are no NaN values after replacing zeros
    if df_processed.isnull().sum().sum() > 0:
        print("\nWarning: There are still NaN values after zero replacement.")
        print(df_processed.isnull().sum())

        # Replace any remaining NaN values with column medians
        for col in df_processed.columns:
            if df_processed[col].isnull().sum() > 0:
                median_value = df_processed[col].median()
                df_processed[col].fillna(median_value, inplace=True)
                print(f"Replaced NaN values in {col} with median: {median_value:.2f}")

    # Identify outliers using IQR method
    print("\nOutlier Analysis:")
    outlier_counts = {}
    for col in df_processed.select_dtypes(include=['float64', 'int64']).columns:
        Q1 = df_processed[col].quantile(0.25)
        Q3 = df_processed[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()
        outlier_counts[col] = outliers
        print(f"Outliers in {col}: {outliers} ({outliers/len(df_processed)*100:.2f}%)")

    # Final check to ensure no NaN values remain
    assert df_processed.isnull().sum().sum() == 0, "There are still NaN values in the processed dataset"

    return df_processed, outlier_counts

In [17]:
# ==================================================================================
# 2.3 Prepare Features and Target Variable
# This function splits data into features and target, standardizes features
# ==================================================================================
def prepare_features_and_target(df_processed):
    """
    Prepare features and target variable for modeling

    Parameters:
    df_processed (pd.DataFrame): Processed diabetes dataset

    Returns:
    tuple: X_train, X_val, X_test, y_train, y_val, y_test, scaler

    Results:
    - Splits data into features (X) and target (y)
    - Analyzes class distribution
    - Splits data into training (70%), validation (15%), and test (15%) sets
    - Standardizes features
    """
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    import numpy as np

    # Define features and target
    X = df_processed.drop('Diabetes', axis=1)
    y = df_processed['Diabetes']

    # Check class balance
    print("Class distribution:")
    print(y.value_counts())
    print(f"Class 0 percentage: {(y == 0).mean()*100:.2f}%")
    print(f"Class 1 percentage: {(y == 1).mean()*100:.2f}%")

    # Split data into train, validation, and test sets
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1765, random_state=42)

    # Check split sizes
    print("\nData split sizes:")
    print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.2f}%)")
    print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.2f}%)")
    print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.2f}%)")

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print("\nFeature standardization complete")
    print(f"Input shape: {X_train_scaled.shape[1]} features")

    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test, scaler

In [18]:
# ==================================================================================
# 2.4 Handle Class Imbalance
# This function checks for and handles class imbalance using SMOTE
# ==================================================================================
def handle_class_imbalance(X_train_scaled, y_train):
    """
    Handle class imbalance if present using SMOTE

    Parameters:
    X_train_scaled (np.array): Standardized training features
    y_train (np.array): Training target variable

    Returns:
    tuple: Balanced X_train_scaled, y_train

    Results:
    - Checks if class imbalance exists
    - Applies SMOTE if needed
    - Reports new class distribution
    """
    # Import SMOTE inside the function to ensure it's defined
    from imblearn.over_sampling import SMOTE

    # Check class distribution
    print("Original class distribution:")
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f"Class {label}: {count} samples ({count/len(y_train)*100:.2f}%)")

    # Check for NaN values before applying SMOTE
    if np.isnan(X_train_scaled).any():
        print("Warning: NaN values detected in training data.")
        print(f"Number of NaN values: {np.isnan(X_train_scaled).sum()}")
        print("Replacing NaN values with column means...")

        # For each column with NaN, replace with column mean
        col_mean = np.nanmean(X_train_scaled, axis=0)
        nan_indices = np.where(np.isnan(X_train_scaled))
        X_train_scaled[nan_indices] = np.take(col_mean, nan_indices[1])

        print("NaN values have been replaced.")

    # Apply SMOTE if class imbalance exists
    if (y_train == 1).mean() < 0.4 or (y_train == 1).mean() > 0.6:
        print("\nApplying SMOTE to balance classes...")
        smote = SMOTE(random_state=42)
        X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

        # Check new distribution
        print("New class distribution after SMOTE:")
        unique, counts = np.unique(y_train_balanced, return_counts=True)
        for label, count in zip(unique, counts):
            print(f"Class {label}: {count} samples ({count/len(y_train_balanced)*100:.2f}%)")

        return X_train_balanced, y_train_balanced
    else:
        print("\nClass distribution is already balanced. SMOTE not applied.")
        return X_train_scaled, y_train

In [19]:
# ==================================================================================
# 3.1 Design Convolutional Neural Network
# This function creates a convolutional neural network architecture
# ==================================================================================
def design_convolutional_network(input_dim):
    """
    Design a deep learning convolution-based network architecture for diabetes prediction

    Parameters:
    input_dim (int): Number of input features

    Returns:
    keras.Model: Compiled neural network model

    Results:
    - Creates a 1D convolutional neural network
    - Prints model summary
    - Explains the architectural choices
    """
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Dense, Dropout, Reshape, Conv1D, MaxPooling1D, Flatten, Input
    from tensorflow.keras.optimizers import Adam

    # Create the model using functional API for more flexibility
    inputs = Input(shape=(input_dim,))

    # Reshape input for 1D convolution (batch_size, n_features, 1)
    x = Reshape((input_dim, 1))(inputs)

    # Convolutional layers
    x = Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Flatten the output from convolutional layers
    x = Flatten()(x)

    # Fully connected layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)

    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)

    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    # Create the model
    model = Model(inputs=inputs, outputs=outputs)

    # Print model summary
    print("Model Architecture:")
    model.summary()

    # Explanation of design choices
    print("\nDesign Choices Justification:")
    print("1. Input reshaping: Converts tabular data into a format suitable for Conv1D layers")
    print("2. Convolutional layers: Extract local patterns from feature sequences")
    print("3. MaxPooling layers: Reduce dimensionality and focus on most important features")
    print("4. Fully connected layers: Integrate learned patterns for classification")
    print("5. ReLU activation: Helps prevent vanishing gradient problem and speeds up training")
    print("6. Dropout layers: Prevents overfitting by randomly deactivating neurons during training")
    print("7. Sigmoid activation in output layer: Appropriate for binary classification, outputs probability")

    return model

In [20]:
# ==================================================================================
# 3.2 Configure Training Process
# This function compiles the model and explains parameter choices
# ==================================================================================
def configure_training_process(model):
    """
    Configure the training process for the neural network

    Parameters:
    model (keras.Model): The neural network model

    Returns:
    keras.Model: Compiled model

    Results:
    - Compiles the model with appropriate loss function and metrics
    - Explains the compilation parameter choices
    """
    from tensorflow.keras.optimizers import Adam

    # Compile the model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    print("\nCompilation Choices:")
    print("1. Adam optimizer: Adaptive learning rate algorithm that handles sparse gradients well")
    print("2. Learning rate of 0.001: Standard starting point that balances learning speed and stability")
    print("3. Binary cross-entropy loss: Standard loss function for binary classification problems")
    print("4. Accuracy metric: Intuitive metric for classification performance evaluation")
    print("5. L2 regularization: Added in Dense layers to prevent overfitting")

    return model

In [21]:
# ==================================================================================
# 4.1 Train Neural Network
# This function trains the model with appropriate callbacks
# ==================================================================================
def train_neural_network(model, X_train, y_train, X_val, y_val):
    """
    Train the neural network model with appropriate callbacks

    Parameters:
    model (keras.Model): Neural network model
    X_train (np.array): Training features
    y_train (np.array): Training target
    X_val (np.array): Validation features
    y_val (np.array): Validation target

    Returns:
    tuple: Trained model and training history

    Results:
    - Trains the model with early stopping, model checkpointing and learning rate scheduling
    - Monitors training and validation metrics
    - Reports final training performance
    """
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
    import matplotlib.pyplot as plt
    import numpy as np

    # Define callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    model_checkpoint = ModelCheckpoint(
        'best_diabetes_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )

    # Train the model
    print("Training the model...")
    history = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stopping, model_checkpoint, reduce_lr],
        verbose=1
    )

    # Report final training performance
    print("\nTraining Performance Analysis:")
    print(f"Final training loss: {history.history['loss'][-1]:.4f}")
    print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Final validation loss: {history.history['val_loss'][-1]:.4f}")
    print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

    best_epoch = np.argmin(history.history['val_loss']) + 1
    print(f"Best model found at epoch {best_epoch}")

    # Check for overfitting
    if history.history['loss'][-1] < history.history['val_loss'][-1]:
        gap = history.history['val_loss'][-1] - history.history['loss'][-1]
        print(f"Model shows signs of overfitting. Gap between training and validation loss: {gap:.4f}")
    else:
        print("Model does not show signs of overfitting.")

    return model, history

In [22]:
# ==================================================================================
# 4.2 Visualize Training Process
# This function creates plots of training and validation metrics
# ==================================================================================
def visualize_training_process(history):
    """
    Visualize the model training process

    Parameters:
    history: Training history from model.fit()

    Results:
    - Plots training and validation loss
    - Plots training and validation accuracy
    - Saves visualization to files
    """
    import matplotlib.pyplot as plt

    # Visualize the training process
    plt.figure(figsize=(12, 5))

    # Plot training & validation loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Plot training & validation accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

    print("Training visualization saved as 'training_history.png'")

In [23]:
# ==================================================================================
# 4.3 Tune Hyperparameters
# This function performs hyperparameter tuning for the model
# ==================================================================================
def tune_hyperparameters(X_train, y_train, X_val, y_val, input_dim):
    """
    Tune hyperparameters for the convolutional neural network model

    Parameters:
    X_train (np.array): Training features
    y_train (np.array): Training target
    X_val (np.array): Validation features
    y_val (np.array): Validation target
    input_dim (int): Number of input features

    Returns:
    dict: Best hyperparameters

    Results:
    - Tests different combinations of hyperparameters
    - Reports best configuration found
    """
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Dense, Dropout, Reshape, Conv1D, MaxPooling1D, Flatten, Input
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping
    import numpy as np

    print("Performing hyperparameter tuning...")

    # Define hyperparameter options
    #learning_rates = [0.01, 0.001, 0.0001]
    learning_rates = [0.01, 0.001]
    #batch_sizes = [16, 32, 64]
    batch_sizes = [16, 32]
    #dropout_rates = [0.2, 0.3, 0.4]
    dropout_rates = [0.2]
    filter_numbers = [32, 64]
    #kernel_sizes = [2, 3, 5]
    kernel_sizes = [3]

    # Store results
    results = []

    # Grid search
    for lr in learning_rates:
        for batch_size in batch_sizes:
            for dropout in dropout_rates:
                for filters in filter_numbers:
                    for kernel_size in kernel_sizes:
                        print(f"\nTesting: LR={lr}, Batch={batch_size}, Dropout={dropout}, Filters={filters}, Kernel={kernel_size}")

                        # Create model with current hyperparameters
                        inputs = Input(shape=(input_dim,))
                        x = Reshape((input_dim, 1))(inputs)
                        x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu')(x)
                        x = MaxPooling1D(pool_size=2)(x)
                        x = Conv1D(filters=filters//2, kernel_size=kernel_size, padding='same', activation='relu')(x)
                        x = MaxPooling1D(pool_size=2)(x)
                        x = Flatten()(x)
                        x = Dense(64, activation='relu')(x)
                        x = Dropout(dropout)(x)
                        x = Dense(32, activation='relu')(x)
                        x = Dropout(dropout)(x)
                        outputs = Dense(1, activation='sigmoid')(x)
                        model = Model(inputs=inputs, outputs=outputs)

                        model.compile(
                            optimizer=Adam(learning_rate=lr),
                            loss='binary_crossentropy',
                            metrics=['accuracy']
                        )

                        # Train with early stopping
                        early_stopping = EarlyStopping(
                            monitor='val_loss',
                            patience=5,
                            restore_best_weights=True,
                            verbose=0
                        )

                        history = model.fit(
                            X_train, y_train,
                            epochs=30,
                            batch_size=batch_size,
                            validation_data=(X_val, y_val),
                            callbacks=[early_stopping],
                            verbose=0
                        )

                        # Get best validation loss
                        best_val_loss = min(history.history['val_loss'])
                        best_val_acc = max(history.history['val_accuracy'])

                        print(f"Best validation loss: {best_val_loss:.4f}")
                        print(f"Best validation accuracy: {best_val_acc:.4f}")

                        # Store results
                        results.append({
                            'lr': lr,
                            'batch_size': batch_size,
                            'dropout': dropout,
                            'filters': filters,
                            'kernel_size': kernel_size,
                            'val_loss': best_val_loss,
                            'val_acc': best_val_acc
                        })

    # Find best hyperparameters
    best_result = min(results, key=lambda x: x['val_loss'])

    print("\nBest Hyperparameters:")
    print(f"Learning Rate: {best_result['lr']}")
    print(f"Batch Size: {best_result['batch_size']}")
    print(f"Dropout Rate: {best_result['dropout']}")
    print(f"Filters: {best_result['filters']}")
    print(f"Kernel Size: {best_result['kernel_size']}")
    print(f"Validation Loss: {best_result['val_loss']:.4f}")
    print(f"Validation Accuracy: {best_result['val_acc']:.4f}")

    return best_result

In [24]:
# ==================================================================================
# 4.4 Evaluate Model on Test Data
# This function evaluates the trained model on the test dataset
# ==================================================================================
def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on the test dataset

    Parameters:
    model (keras.Model): Trained neural network model
    X_test (np.array): Test features
    y_test (np.array): Test target

    Results:
    - Generates classification report
    - Creates confusion matrix
    - Plots ROC curve and calculates AUC
    - Reports detailed metrics
    """
    from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns

    # Predict on test data
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)

    # Calculate metrics
    print("\nModel Evaluation on Test Data:")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig('confusion_matrix.png')
    plt.close()
    print("Confusion matrix saved as 'confusion_matrix.png'")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.savefig('roc_curve.png')
    plt.close()
    print("ROC curve saved as 'roc_curve.png'")

    print(f"\nROC AUC: {roc_auc:.4f}")

    # Additional metrics
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    f1 = 2 * (precision * recall) / (precision + recall)

    print("\nDetailed Metrics:")
    print(f"Accuracy: {(tp + tn) / (tp + tn + fp + fn):.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall (Sensitivity): {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score: {f1:.4f}")

In [25]:
# ==================================================================================
# Main Function: Run Complete Workflow
# This function runs the entire diabetes prediction workflow
# ==================================================================================
def run_diabetes_prediction_workflow(file_path):
   """
   Run the complete diabetes prediction workflow

   Parameters:
   file_path (str): Path to the diabetes dataset

   Returns:
   keras.Model: Trained and optimized model

   Results:
   - Executes all steps of the workflow
   - Reports progress and results for each step
   """
   import numpy as np
   import tensorflow as tf
   from tensorflow.keras.models import Model
   from tensorflow.keras.layers import Dense, Dropout, Reshape, Conv1D, MaxPooling1D, Flatten, Input
   from tensorflow.keras.optimizers import Adam

   # Set random seeds for reproducibility
   np.random.seed(42)
   tf.random.set_seed(42)

   print("="*80)
   print("DIABETES PREDICTION USING CONVOLUTIONAL NEURAL NETWORKS")
   print("="*80)

   # Step 1: Import libraries and load data
   print("\nStep 1: Loading data...")
   df = import_libraries_and_load_data(file_path)

   # Step 2.1: Data exploration
   print("\nStep 2.1: Performing exploratory data analysis...")
   feature_importance = create_eda_visualizations(df)

   # Step 2.2: Analyze data quality
   print("\nStep 2.2: Analyzing data quality...")
   df_processed, outlier_counts = analyze_data_quality(df)

   # Step 2.3: Prepare features and target
   print("\nStep 2.3: Preparing features and target...")
   X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_features_and_target(df_processed)

   # Step 2.4: Handle class imbalance
   print("\nStep 2.4: Handling class imbalance...")
   X_train_balanced, y_train_balanced = handle_class_imbalance(X_train, y_train)

   # Step 3.1: Neural network design
   print("\nStep 3.1: Designing convolutional neural network...")
   input_dim = X_train.shape[1]
   model = design_convolutional_network(input_dim)

   # Step 3.2: Configure training process
   print("\nStep 3.2: Configuring training process...")
   model = configure_training_process(model)

   # Step 4.1: Train with default hyperparameters first
   print("\nStep 4.1: Training model with default hyperparameters...")
   model, history = train_neural_network(model, X_train_balanced, y_train_balanced, X_val, y_val)

   # Step 4.2: Visualize training process
   print("\nStep 4.2: Visualizing training process...")
   visualize_training_process(history)

   # Step 4.3: Tune hyperparameters
   print("\nStep 4.3: Tuning hyperparameters...")
   best_params = tune_hyperparameters(X_train_balanced, y_train_balanced, X_val, y_val, input_dim)

   # Step 4.4: Create optimized model with best hyperparameters
   print("\nStep 4.4: Creating optimized model with best hyperparameters...")
   inputs = Input(shape=(input_dim,))
   x = Reshape((input_dim, 1))(inputs)
   x = Conv1D(filters=best_params['filters'], kernel_size=best_params['kernel_size'],
             padding='same', activation='relu')(x)
   x = MaxPooling1D(pool_size=2)(x)
   x = Conv1D(filters=best_params['filters']//2, kernel_size=best_params['kernel_size'],
             padding='same', activation='relu')(x)
   x = MaxPooling1D(pool_size=2)(x)
   x = Flatten()(x)
   x = Dense(64, activation='relu')(x)
   x = Dropout(best_params['dropout'])(x)
   x = Dense(32, activation='relu')(x)
   x = Dropout(best_params['dropout'])(x)
   outputs = Dense(1, activation='sigmoid')(x)
   optimized_model = Model(inputs=inputs, outputs=outputs)

   optimized_model.compile(
       optimizer=Adam(learning_rate=best_params['lr']),
       loss='binary_crossentropy',
       metrics=['accuracy']
   )

   # Step 4.5: Train the optimized model
   print("\nStep 4.5: Training optimized model...")
   optimized_model, optimized_history = train_neural_network(
       optimized_model,
       X_train_balanced, y_train_balanced,
       X_val, y_val
   )

   # Step 4.6: Visualize optimized training process
   print("\nStep 4.6: Visualizing optimized training process...")
   visualize_training_process(optimized_history)

   # Step 4.7: Evaluate the model on test data
   print("\nStep 4.7: Evaluating model on test data...")
   evaluate_model(optimized_model, X_test, y_test)

   print("\nWorkflow completed successfully!")

   # Compare initial model to optimized model
   print("\nComparison of Initial vs. Optimized Model:")
   print(f"Initial Validation Loss: {min(history.history['val_loss']):.4f}")
   print(f"Initial Validation Accuracy: {max(history.history['val_accuracy']):.4f}")
   print(f"Optimized Validation Loss: {min(optimized_history.history['val_loss']):.4f}")
   print(f"Optimized Validation Accuracy: {max(optimized_history.history['val_accuracy']):.4f}")

   improvement_loss = min(history.history['val_loss']) - min(optimized_history.history['val_loss'])
   improvement_acc = max(optimized_history.history['val_accuracy']) - max(history.history['val_accuracy'])

   print(f"Loss Improvement: {improvement_loss:.4f} ({improvement_loss/min(history.history['val_loss'])*100:.2f}%)")
   print(f"Accuracy Improvement: {improvement_acc:.4f} ({improvement_acc/max(history.history['val_accuracy'])*100:.2f}%)")

   return optimized_model

In [ ]:
model = run_diabetes_prediction_workflow('/content/sample_data/enhanced_diabetes_dataset.csv')

DIABETES PREDICTION USING CONVOLUTIONAL NEURAL NETWORKS

Step 1: Loading data...
First 5 rows of the dataset:
   Age  Gender  Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin  \
0   52       1            1      124           95.0           20.0        0   
1   43       1            2      144           86.0           23.0        3   
2   55       1            4      141          113.0           18.0      297   
3   68       1            3      125          110.0           32.0      210   
4   41       0            0      177           92.0           14.0      189   

    BMI  DiabetesPedigreeFunction  HbA1c  FastingBS  Triglycerides   HDL  \
0  36.0                     0.078    4.7         87          142.0  37.0   
1  32.9                     0.118    6.3        135          162.0  89.0   
2  43.4                     0.139    5.4        113          180.0  45.0   
3  22.7                     0.197    5.2         93          262.0  56.0   
4  38.1                     0.078  

<ipython-input-16-6062dc7ddc5a>:33: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_processed[col].fillna(median_value, inplace=True)


Model Architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 16, 1)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 16, 64)              │             256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 8, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 8, 32)               │           6,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 4, 32)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 16,801 (65.63 KB)

 Trainable params: 16,801 (65.63 KB)

 Non-trainable params: 0 (0.00 B)


Design Choices Justification:
1. Input reshaping: Converts tabular data into a format suitable for Conv1D layers
2. Convolutional layers: Extract local patterns from feature sequences
3. MaxPooling layers: Reduce dimensionality and focus on most important features
4. Fully connected layers: Integrate learned patterns for classification
5. ReLU activation: Helps prevent vanishing gradient problem and speeds up training
6. Dropout layers: Prevents overfitting by randomly deactivating neurons during training
7. Sigmoid activation in output layer: Appropriate for binary classification, outputs probability

Step 3.2: Configuring training process...

Compilation Choices:
1. Adam optimizer: Adaptive learning rate algorithm that handles sparse gradients well
2. Learning rate of 0.001: Standard starting point that balances learning speed and stability
3. Binary cross-entropy loss: Standard loss function for binary classification problems
4. Accuracy metric: Intuitive metric for classification 

70/70 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.5836 - loss: 0.6665 - val_accuracy: 0.7209 - val_loss: 0.5042 - learning_rate: 0.0010
Epoch 2/100
62/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6727 - loss: 0.5831
Epoch 2: val_loss did not improve from 0.50420
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6763 - loss: 0.5818 - val_accuracy: 0.6811 - val_loss: 0.5549 - learning_rate: 0.0010
Epoch 3/100
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7224 - loss: 0.5611
Epoch 3: val_loss did not improve from 0.50420
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7226 - loss: 0.5609 - val_accuracy: 0.6910 - val_loss: 0.5199 - learning_rate: 0.0010
Epoch 4/100
68/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7354 - loss: 0.5341
Epoch 4: val_loss did not improve from 0.50420
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7361 - loss: 0.5337 - val_accuracy: 0.7043 - val_loss: 0.5282 - learning_rate: 0.0010
Epoch 5/100
68/70 ━━━━━━━━━━━━━━━━━━━━ 0s 7m